# Village Population Intelligence
## Identifying High-Priority Rural Regions Using Demographic Data

**Project type:** Data Science / Exploratory Data Analysis / Priority Ranking

### Objective
Use village-level demographic and service-access indicators to identify rural regions that may need additional planning attention. The notebook cleans data, creates indicators, calculates a transparent composite priority score, ranks villages, and visualizes the results.

### Important data note
This notebook runs immediately with a **clearly labelled illustrative demo dataset** if you do not provide a CSV. Demo values are synthetic and must not be presented as actual Census of India statistics. For a real submission, replace the demo data with a reliable village-level dataset and cite its source.

### Expected CSV columns
`Village`, `District`, `State`, `Population`, `Households`, `Population_Density`, `Literacy_Rate`, `Female_Male_Ratio`, `SC_ST_Pct`, `Water_Access_Pct`, `Healthcare_Distance_km`, `School_Distance_km`, `Employment_Rate`

Rates should be numeric percentages (0–100); `Female_Male_Ratio` is females per 1,000 males. Missing optional indicators are handled conservatively, but the core `Village`, `District`, `State`, and `Population` fields are required.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 1. Load the demographic dataset

Set `CSV_PATH` to the location of your real village-level CSV file. If the file is not found, the notebook uses a synthetic dataset for demonstration so the analysis can still be explored end-to-end.

In [ ]:
CSV_PATH = "village_demographic_data.csv"

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
    DATA_MODE = "USER-PROVIDED CSV"
else:
    # Synthetic illustration only — not official census data.
    rng = np.random.default_rng(42)
    villages = [
        ("Rampur", "Pune", "Maharashtra"), ("Dhanora", "Nagpur", "Maharashtra"),
        ("Khedgaon", "Nashik", "Maharashtra"), ("Bori", "Amravati", "Maharashtra"),
        ("Pimpalwadi", "Ahmednagar", "Maharashtra"), ("Sonwadi", "Satara", "Maharashtra"),
        ("Chandpur", "Bhandara", "Maharashtra"), ("Nandgaon", "Jalgaon", "Maharashtra"),
        ("Devli", "Wardha", "Maharashtra"), ("Karanji", "Yavatmal", "Maharashtra"),
        ("Bhavani", "Indore", "Madhya Pradesh"), ("Lakshmipur", "Bhopal", "Madhya Pradesh"),
        ("Sundarpur", "Raipur", "Chhattisgarh"), ("Jaitpur", "Bilaspur", "Chhattisgarh"),
        ("Shivnagar", "Jaipur", "Rajasthan"), ("Gopalpura", "Udaipur", "Rajasthan"),
        ("Haripur", "Patna", "Bihar"), ("Madhopur", "Gaya", "Bihar"),
        ("Krishnapur", "Nadia", "West Bengal"), ("Anandgram", "Birbhum", "West Bengal"),
        ("Chinnapalli", "Anantapur", "Andhra Pradesh"), ("Kothuru", "Visakhapatnam", "Andhra Pradesh"),
        ("Belahalli", "Ballari", "Karnataka"), ("Hosahalli", "Mysuru", "Karnataka"),
        ("Perumalpatti", "Madurai", "Tamil Nadu"), ("Vellankulam", "Tirunelveli", "Tamil Nadu"),
        ("Keshopur", "Lucknow", "Uttar Pradesh"), ("Basantpur", "Gorakhpur", "Uttar Pradesh"),
        ("Chakpara", "Kamrup", "Assam"), ("Sonapur", "Dibrugarh", "Assam")
    ]
    rows = []
    for village, district, state in villages:
        population = int(rng.integers(650, 8500))
        households = max(80, int(population / rng.uniform(4.0, 5.8)))
        rows.append({
            "Village": village, "District": district, "State": state,
            "Population": population, "Households": households,
            "Population_Density": int(rng.integers(90, 950)),
            "Literacy_Rate": round(float(rng.uniform(48, 96)), 1),
            "Female_Male_Ratio": int(rng.integers(850, 1080)),
            "SC_ST_Pct": round(float(rng.uniform(3, 72)), 1),
            "Water_Access_Pct": round(float(rng.uniform(35, 100)), 1),
            "Healthcare_Distance_km": round(float(rng.uniform(1, 35)), 1),
            "School_Distance_km": round(float(rng.uniform(0.5, 15)), 1),
            "Employment_Rate": round(float(rng.uniform(35, 92)), 1)
        })
    df_raw = pd.DataFrame(rows)
    DATA_MODE = "SYNTHETIC DEMO DATA"

print(f"Data mode: {DATA_MODE}")
print(f"Rows: {len(df_raw):,} | Columns: {len(df_raw.columns)}")
display(df_raw.head())


## 2. Data cleaning and preprocessing

- Standardize column names and text fields.
- Check required columns.
- Convert demographic indicators to numeric values.
- Remove duplicate village records and invalid population rows.
- Fill missing indicator values with the median of the available data, and report missingness before filling.


In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()

required = ["Village", "District", "State", "Population"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"CSV is missing required columns: {missing_required}")

text_cols = ["Village", "District", "State"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

numeric_candidates = [
    "Population", "Households", "Population_Density", "Literacy_Rate",
    "Female_Male_Ratio", "SC_ST_Pct", "Water_Access_Pct",
    "Healthcare_Distance_km", "School_Distance_km", "Employment_Rate"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values before cleaning:")
display(df.isna().sum().rename("Missing values").to_frame())

df = df.drop_duplicates(subset=["Village", "District", "State"])
df = df.dropna(subset=["Village", "District", "State", "Population"])
df = df[df["Population"] > 0].copy()

# Optional indicators: if absent, create them as missing; they are imputed below.
optional_features = [
    "Households", "Population_Density", "Literacy_Rate", "Female_Male_Ratio",
    "SC_ST_Pct", "Water_Access_Pct", "Healthcare_Distance_km",
    "School_Distance_km", "Employment_Rate"
]
for col in optional_features:
    if col not in df.columns:
        df[col] = np.nan

# Derived density/household estimate where possible.
if df["Population_Density"].isna().all():
    df["Population_Density"] = np.nan
if df["Households"].isna().all():
    df["Households"] = np.nan

for col in optional_features:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].median())
    else:
        # neutral placeholder if a feature is wholly unavailable
        df[col] = 0.0

print(f"Cleaned records: {len(df):,}")
display(df.head())


## 3. Exploratory data analysis

Review the overall population distribution and compare selected indicators. These charts describe the dataset; they do not by themselves establish why a village has a particular outcome.

In [ ]:
print("Dataset overview")
print(f"Villages analysed: {len(df):,}")
print(f"States represented: {df['State'].nunique():,}")
print(f"Districts represented: {df['District'].nunique():,}")
print(f"Total population across records: {df['Population'].sum():,.0f}")
display(df[["Population", "Literacy_Rate", "Water_Access_Pct",
            "Healthcare_Distance_km", "Employment_Rate"]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["Population"], bins=12, edgecolor="black")
axes[0].set_title("Village Population Distribution")
axes[0].set_xlabel("Population")
axes[0].set_ylabel("Number of villages")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

state_population = df.groupby("State")["Population"].sum().sort_values(ascending=False).head(10)
axes[1].barh(state_population.index[::-1], state_population.values[::-1])
axes[1].set_title("Population in Records by State (Top 10)")
axes[1].set_xlabel("Combined population of listed villages")
axes[1].set_ylabel("State")
axes[1].grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 4. Feature engineering and priority scoring

The project creates a **Rural Priority Score (0–100)**. Higher scores indicate greater potential need based on the available indicators.

### Scoring criteria and weights
- **Low literacy rate — 20%**
- **Low household water access — 25%**
- **Long distance to healthcare — 20%**
- **Long distance to school — 15%**
- **Low employment rate — 15%**
- **Population scale — 5%**

Each criterion is min-max normalized to a 0–1 scale. For access, literacy, and employment, lower values mean higher need, so the direction is reversed. Population is given a small weight so that large population alone does not dominate service-access needs.

**Caution:** weights are illustrative and should be agreed with local experts or project stakeholders. This is a screening model, not an official government classification. If an indicator is unavailable in your real data, collect it or revise the weights rather than treating the placeholder as real evidence.


In [ ]:
# Direction-adjusted indicators: higher value = higher estimated need.
df["Low_Literacy_Need"] = 100 - df["Literacy_Rate"].clip(0, 100)
df["Low_Water_Access_Need"] = 100 - df["Water_Access_Pct"].clip(0, 100)
df["Healthcare_Access_Need"] = df["Healthcare_Distance_km"].clip(lower=0)
df["School_Access_Need"] = df["School_Distance_km"].clip(lower=0)
df["Low_Employment_Need"] = 100 - df["Employment_Rate"].clip(0, 100)
df["Population_Scale"] = df["Population"].clip(lower=0)

score_features = [
    "Low_Literacy_Need", "Low_Water_Access_Need", "Healthcare_Access_Need",
    "School_Access_Need", "Low_Employment_Need", "Population_Scale"
]
weights = np.array([0.20, 0.25, 0.20, 0.15, 0.15, 0.05])

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[score_features])
df["Rural_Priority_Score"] = scaled @ weights * 100

df = df.sort_values("Rural_Priority_Score", ascending=False).reset_index(drop=True)
df["Priority_Rank"] = np.arange(1, len(df) + 1)

def priority_band(score):
    if score >= 67:
        return "High"
    elif score >= 34:
        return "Medium"
    return "Lower"

df["Priority_Level"] = df["Rural_Priority_Score"].apply(priority_band)

top_10 = df.head(10)
display(top_10[[
    "Priority_Rank", "Village", "District", "State", "Population",
    "Literacy_Rate", "Water_Access_Pct", "Healthcare_Distance_km",
    "School_Distance_km", "Employment_Rate", "Rural_Priority_Score", "Priority_Level"
]].round(2))


## 5. Identify high-priority villages and summarize findings

The highest-ranked records are candidates for further investigation. The ranking should be checked against field data, data quality, and local context before any resource-allocation decision.

In [ ]:
high_priority = df[df["Priority_Level"] == "High"].copy()

print("=" * 78)
print("VILLAGE POPULATION INTELLIGENCE — PRIORITY SUMMARY")
print("=" * 78)
print(f"Villages assessed                 : {len(df):,}")
print(f"High-priority villages            : {len(high_priority):,}")
print(f"Total population in high-priority records: {high_priority['Population'].sum():,.0f}")
if len(df):
    top = df.iloc[0]
    print(f"Highest-priority village          : {top['Village']}, {top['District']}, {top['State']}")
    print(f"Priority score                    : {top['Rural_Priority_Score']:.2f}/100")
    print(f"Population                        : {top['Population']:,.0f}")
    print(f"Water access                      : {top['Water_Access_Pct']:.1f}%")
    print(f"Literacy rate                     : {top['Literacy_Rate']:.1f}%")
    print(f"Healthcare distance               : {top['Healthcare_Distance_km']:.1f} km")
print("=" * 78)

print("\nPriority-level distribution:")
display(df["Priority_Level"].value_counts().reindex(["High", "Medium", "Lower"], fill_value=0).rename("Village count").to_frame())

print("\nPotential focus areas among the top 10 (dataset averages):")
display(top_10[["Literacy_Rate", "Water_Access_Pct", "Healthcare_Distance_km",
                "School_Distance_km", "Employment_Rate"]].mean().rename("Top-10 average").to_frame().round(2))


## 6. Visualization dashboard

- Chart 1 compares the top 15 villages by priority score.
- Chart 2 shows population versus priority score; labels identify the most highly ranked records.
- Chart 3 compares service and demographic indicators for the top 10 villages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_top = df.head(min(15, len(df))).sort_values("Rural_Priority_Score", ascending=True)
bar_colors = ["#c0392b" if level == "High" else "#2980b9" if level == "Medium" else "#7f8c8d"
              for level in plot_top["Priority_Level"]]
axes[0].barh(plot_top["Village"] + " (" + plot_top["District"] + ")",
             plot_top["Rural_Priority_Score"], color=bar_colors)
axes[0].set_title("Highest-Priority Rural Villages")
axes[0].set_xlabel("Rural Priority Score (0–100)")
axes[0].set_ylabel("Village (District)")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

scatter = axes[1].scatter(df["Population"], df["Rural_Priority_Score"],
                          c=df["Rural_Priority_Score"], s=70, alpha=0.75,
                          edgecolors="black", linewidths=0.4)
axes[1].set_title("Population vs. Rural Priority Score")
axes[1].set_xlabel("Village Population")
axes[1].set_ylabel("Rural Priority Score (0–100)")
axes[1].grid(True, linestyle="--", alpha=0.4)
for _, row in df.head(min(5, len(df))).iterrows():
    axes[1].annotate(row["Village"], (row["Population"], row["Rural_Priority_Score"]),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
fig.colorbar(scatter, ax=axes[1], label="Priority score")
plt.tight_layout()
plt.show()

# Indicator profile for the top 10 villages; all indicators shown on a 0–100 scale.
profile_cols = ["Literacy_Rate", "Water_Access_Pct", "Employment_Rate"]
profile = top_10.set_index("Village")[profile_cols].copy()
if not profile.empty:
    ax = profile.plot(kind="bar", figsize=(14, 5))
    ax.set_title("Literacy, Water Access and Employment: Top 10 Priority Villages")
    ax.set_ylabel("Percent (%)")
    ax.set_xlabel("Village")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


## 7. Additional graphs for priority insights
These graphs summarize how villages are distributed across priority levels and how key demographic/service indicators relate to the Rural Priority Score. They are descriptive and should be interpreted alongside data quality and local context.


In [ ]:
# Graph 4: Distribution of villages by priority level
priority_counts = df["Priority_Level"].value_counts().reindex(["High", "Medium", "Lower"], fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(priority_counts.index, priority_counts.values, edgecolor="black")
axes[0].set_title("Number of Villages by Priority Level")
axes[0].set_xlabel("Priority Level")
axes[0].set_ylabel("Number of Villages")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
for i, value in enumerate(priority_counts.values):
    axes[0].text(i, value, str(int(value)), ha="center", va="bottom")

# Graph 5: Water access versus priority score
axes[1].scatter(df["Water_Access_Pct"], df["Rural_Priority_Score"],
                s=75, alpha=0.75, edgecolors="black", linewidths=0.4)
axes[1].set_title("Household Water Access vs. Rural Priority Score")
axes[1].set_xlabel("Household Water Access (%)")
axes[1].set_ylabel("Rural Priority Score (0–100)")
axes[1].grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Graph 6: Correlation matrix for numerical indicators (Matplotlib only)
corr_cols = ["Population", "Literacy_Rate", "Water_Access_Pct",
             "Healthcare_Distance_km", "School_Distance_km",
             "Employment_Rate", "Rural_Priority_Score"]
corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Correlation Between Demographic Indicators and Priority Score")
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Correlation coefficient")
plt.tight_layout()
plt.show()


## 8. Export the ranked results

The CSV can be used in a report or presentation. When using demo data, keep the synthetic-data note with any exported results.

In [ ]:
output_file = "village_priority_ranking.csv"
export_cols = [
    "Priority_Rank", "Village", "District", "State", "Population", "Households",
    "Population_Density", "Literacy_Rate", "Female_Male_Ratio", "SC_ST_Pct",
    "Water_Access_Pct", "Healthcare_Distance_km", "School_Distance_km",
    "Employment_Rate", "Rural_Priority_Score", "Priority_Level"
]
df[export_cols].to_csv(output_file, index=False)
print(f"Saved ranked results to: {os.path.abspath(output_file)}")


## 9. Conclusion and future scope

### Conclusion
This project demonstrates a repeatable way to transform village-level demographic and service-access data into an interpretable priority shortlist. It combines population scale with indicators of access to water, education, healthcare, literacy, and employment. The ranked output can help planners decide where to investigate needs more closely.

### Limitations
1. The demo dataset is synthetic; it does not describe real villages.
2. Min-max scores are relative to the villages in the selected dataset, so scores can change when records are added or removed.
3. The composite score depends on the chosen indicators and weights.
4. Missing or unreliable source data can distort rankings.
5. A high score indicates a need for investigation, not proof of need or a final funding decision.

### Future scope
- Use verified Census of India village-level data and cite the exact dataset/year.
- Add road connectivity, drinking-water reliability, sanitation, internet access, health-facility capacity, and age-group data where available.
- Compare districts and states using maps or geospatial analysis.
- Validate rankings with field surveys and local administrative feedback.
- Test alternate weights and show how stable the ranking remains.

### Suggested project outcome
**A data-driven shortlist of rural villages that may warrant priority assessment**, supported by transparent scoring, summary tables, visualizations, and an exportable CSV.
